# Federated Learning Pipeline Demo
This notebook outlines a processing pipeline for federated learning. It performs: 
- **DICOM anonymization** 
- **N4 bias‑field correction** 
- **Z‑score intensity normalization**

## Directory Overview

The pipeline starts with a clean copy of the source data and produces two derivative
datasets: one for anonymized DICOMs and one for processed NIfTIs.

### Initial layout

In [ ]:
"""
Pipeline_Workspace/
├── Dataset_True_Copy/ # reference copy
│ ├── DICOM/
│ │ └── …
│ ├── DICOMDIR
│ └── selection.csv # chosen studies
"""

### Final layout

In [ ]:
"""Pipeline_Workspace/
├── Dataset_True_Copy/ # reference copy
│ ├── DICOM/
│ │ └── …
│ ├── DICOMDIR
│ └── selection.csv # chosen studies
│
├── STS_<INSTITUTION>_Dataset/ # anonymized DICOM
│ ├── DICOM/
│ │ └── …
│ └── ledger.csv # source-to-target map
│
└── sts.<INSTITUTION>/ # processed NIfTIs
├── sts001.000001/
│ ├── sts001.000001.t1.nii
│ └── sts001.000001.t2.nii
├── sts001.000002/
│ ├── sts001.000002.t1.nii
│ └── sts001.000002.t2.nii
└── data.csv # cohort-level clinical data
"""

## Metadata Review Prior to Processing

Loading raw DICOM files into 3D Slicer reveals patient‑identifying metadata.  
We will inspect these headers to determine which fields require removal.

## Processing
Running the pipeline below generates the anonymized DICOM set and the processed NIfTI directory described earlier.

In [ ]:
from pipeline import main

main()

## Metadata Review Upon Processing

Loading processed DICOM files into 3D Slicer reveals that the patient‑identifying metadata has been scrubbed.

## Image Review Upon Processing
The visualized output demonstrates that N4 bias-field correction and Z-score normalization were also correctly executed.

In [ ]:
from pathlib import Path
from imaging.imaging_utils import compare_random_images, compare_histograms

dicom = Path(r"C:\Users\josho\OneDrive - McGill University\MGH-SRC\Pipeline Workspace\STS_002_Dataset\DICOM\PA000001\ST000001\SE000005")
nifti = Path(r"C:\Users\josho\OneDrive - McGill University\MGH-SRC\Pipeline Workspace\sts.002\sts.002.000001\sts.002.000001.t1.nii")

compare_random_images(dicom, nifti, seed=9)
compare_histograms(dicom, nifti)

## Managing Sensitive Fields
Users may wish to add to or remove from the fields that have labelled sensitive by default. The code below leverages functions that allow them to accomplish this.

In [ ]:
from dicom.dicom_anonymize import (
    load_sensitive_fields,
    add_sensitive_field,
    remove_sensitive_field,
)

sensitive_fields = Path(r"C:\Users\josho\OneDrive - McGill University\SarcomaAI\python_pipeline\anonymization_targets\sensitive_fields.json")
fields = load_sensitive_fields(sensitive_fields)

print("Initial:")
for field in fields:
    print(f"    {field}")

In [ ]:
add_sensitive_field("EXAMPLE", sensitive_fields)
fields = load_sensitive_fields(sensitive_fields)

print("After addition:")
for field in fields:
    print(f"    {field}")

In [ ]:
remove_sensitive_field("EXAMPLE", sensitive_fields)
fields = load_sensitive_fields(sensitive_fields)

print("After removal:")
for field in fields:
    print(f"    {field}")

## Assumptions and Future Directions
**Directory structure as input:**
The pipeline currently assumes a rigid folder hierarchy, limiting portability across institutions.

**MRN dependence:**
The pipeline requires an identifier to be present to map new MMNN IDs to clinical IDs. MRN was used in this case, making it a requirement.

**Clearing the input between runs:**
A mid‑run crash leaves unprocessed cases that must be tracked and purged manually.

**Python–JavaScript event handling:**
At present, there is no measure for the Python script to communicate with the web‑based GUI, and vice-versa. Exposing Python logs and progress updates would allow JavaScript to subscribe to real‑time emits, enabling interactive monitoring and richer user interfaces.